# IOAI — 2024 Final Stage Anomaly Detection (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/test.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2024-final-stage-anomaly-detection/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터 준비:', sorted(os.listdir('data'))[:8])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# 이상 탐지 (Detekcja anomalii) — Polish AI Olympiad I · 결선

이미지가 **정상(0)** 인지 **이상(1)** 인지 판정한다. 핵심 제약: **학습셋은 정상 이미지만** 담겨 있으므로
(라벨 전부 0), **자기지도/비지도** 방식으로 정상 분포를 학습해 이상을 찾아야 한다.

- `data/train/` + `data/train.csv`(filename,label=0) — 정상 이미지 7000장(64×64)
- `data/test/` + `data/test.csv`(filename) — 균형 테스트 2000장(정상 1000/이상 1000, 라벨 비공개)

**제출**: 각 test 이미지에 대해 `id(=filename),label(0/1)` 을 `submission.csv` 로 저장 → accuracy 채점.

아래 베이스라인은 **합성곱 오토인코더**를 정상 이미지로 학습한 뒤, 재구성 오차가 큰 이미지를 이상으로 판정한다.
전체 폴란드어 원문/규칙은 Overview 탭 참고.

## 데이터 로드

In [ ]:
import os, glob
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
device = "cuda" if torch.cuda.is_available() else "cpu"; print(device)

def load(paths):
    xs = [np.asarray(Image.open(p).convert("RGB").resize((64,64)), dtype=np.float32)/255. for p in paths]
    return torch.tensor(np.stack(xs)).permute(0,3,1,2)   # (N,3,64,64)

train_files = ["data/train/"+f for f in pd.read_csv("data/train.csv")["filename"]]
test_df = pd.read_csv("data/test.csv"); test_files = ["data/test/"+f for f in test_df["filename"]]
Xtr = load(train_files); Xte = load(test_files)
print("train", Xtr.shape, "test", Xte.shape)

## 오토인코더 (정상 이미지로 학습)

In [ ]:
class AE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(nn.Conv2d(3,32,3,2,1), nn.ReLU(), nn.Conv2d(32,64,3,2,1), nn.ReLU(),
                                 nn.Conv2d(64,128,3,2,1), nn.ReLU())        # 64->8
        self.dec = nn.Sequential(nn.ConvTranspose2d(128,64,4,2,1), nn.ReLU(),
                                 nn.ConvTranspose2d(64,32,4,2,1), nn.ReLU(),
                                 nn.ConvTranspose2d(32,3,4,2,1), nn.Sigmoid())
    def forward(self,x): return self.dec(self.enc(x))

ae = AE().to(device); opt = torch.optim.Adam(ae.parameters(), 1e-3); crit = nn.MSELoss()
dl = DataLoader(Xtr, batch_size=128, shuffle=True)
for ep in range(8):
    ae.train(); tot=0
    for xb in dl:
        xb=xb.to(device); opt.zero_grad(); loss=crit(ae(xb), xb); loss.backward(); opt.step(); tot+=loss.item()*len(xb)
    print(f"epoch {ep} mse {tot/len(Xtr):.5f}")

## 재구성 오차 임계값 → submission.csv

In [ ]:
@torch.no_grad()
def recon_err(X):
    ae.eval(); errs=[]
    for i in range(0,len(X),256):
        xb=X[i:i+256].to(device); e=((ae(xb)-xb)**2).mean(dim=[1,2,3]).cpu().numpy(); errs.append(e)
    return np.concatenate(errs)

tr_err = recon_err(Xtr); te_err = recon_err(Xte)
# 정상만으로 학습한 오차 분포의 상위 백분위를 임계값으로(비지도) — 오차 큰 이미지를 이상(1)으로
thr = np.percentile(tr_err, 80)
pred = (te_err > thr).astype(int)
pd.DataFrame({"id": test_df["filename"], "label": pred}).to_csv("submission.csv", index=False)
print("saved submission.csv", len(pred), "| anomaly ratio", pred.mean().round(3), "| thr", round(float(thr),5))

더 끌어올리려면: 더 깊은 AE·SSIM 손실·앙상블·임계값 정교화(정상 오차 분포 기반)를 시도하세요.

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)